In [ ]:
import os

# The notebook lives in research/ but the paths below ("Data/") are
# relative to the repo root, so step up one level when needed.
if os.path.basename(os.getcwd()) == "research":
    os.chdir("..")

os.getcwd()

In [55]:
%pwd

'/Users/kanishkachandrakar/Desktop/End-to-end-Medical-Chatbot-Generative-AI'

In [56]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [57]:
#Extract data from the PDF
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [58]:
extracted_data = load_pdf("Data/")

In [59]:
#extracted_data

In [60]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [61]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 5860


In [62]:
#text_chunks

In [63]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [64]:
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [65]:
embeddings = download_hugging_face_embeddings()

In [66]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [67]:
#query_result

In [72]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os
from dotenv import load_dotenv

load_dotenv()

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

index_name = "medicalbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

In [69]:
from dotenv import load_dotenv

load_dotenv()  # loads PINECONE_API_KEY / GROQ_API_KEY from .env

In [70]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

In [71]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [73]:
docsearch

In [74]:
retriever = docsearch.as_retriever(search_type='similarity', search_kwargs={"k":3})

In [75]:
retrieved_docs = retriever.invoke("What is Acne?")

In [76]:
retrieved_docs

[Document(id='846e3f0b-3aa9-41bb-9c32-4bd5c6910c0d', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data/Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='bcf8cf65-a170-4475-a6ca-26ce248e2802', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data/Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='77a59ba3-0248-4c5d-9284-143b4c3a1138', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:

In [77]:
from langchain_groq import ChatGroq
llm = ChatGroq(temperature=0, groq_api_key=os.environ.get("GROQ_API_KEY"), model_name="deepseek-r1-distill-qwen-32b")


#from langchain_openai import OpenAI
#llm = OpenAI(temperature=0.4, max_tokens=500)

In [103]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, google the "
    "terms and give the answer. Use three sentences maximum and keep the "
    "answer concise. Write it as definititions, do not include your own words or personal thoughts. "
    "Give answer only for medical terms. If not medical terms, mention that it is not a medical term, do not answer, and just say that you cant help"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [104]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [105]:
response = rag_chain.invoke({"input": "What is a big bowl?"})
answer = response["answer"]

import re
cleaned_answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()

print(cleaned_answer)

I cannot help with that as "big bowl" is not a medical term.
